# Week 1 Day 2: comparing three named free claims agents

Begetterz

Which named free model gets the most claims right with the lab's agent?
This notebook uses the same five `tiny` queue claims, instructions, policy wording
and two tools as `labs/week1day2free.ipynb`. Only the model changes.
A correct answer matches the handler's decision and payout within $1, inclusive.

**Live calls are off by default.** Run all cells to inspect `results.csv` and run
the offline checks. To collect or resume measurements, set `RUN_LIVE = True`
below and run all cells using the repository's `.venv` kernel. The root `.env`
supplies `OPENROUTER_API_KEY`. No paid model or fallback model is used.

In [ ]:
import csv
import hashlib
import json
import os
import time
from contextlib import redirect_stdout
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import Literal

import httpx
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, wrap_model_call
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import APIError
from pydantic import Field

RUN_LIVE = False
MODELS = (
    "nvidia/nemotron-3.5-super-vl:free",
    "dots-studio/dots-3-note-preview:free",
    "liquid/lfm-2.5-2.6b:free",
)
MAX_MODEL_CALLS = 5
REQUEST_INTERVAL = 4.1
EXPERIMENT_VERSION = 1
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "data/claims.json").is_file() and (path / "pyproject.toml").is_file()
)
FOLDER = ROOT / "community_contributions/Begetterz/week1day2"
RESULTS_PATH = FOLDER / "results.csv"
load_dotenv(ROOT / ".env")

## Fixed inputs

The handler labels select the lab's five sample claims and score answers locally.
They are never passed to the agent or its tools. The held-out set and claims
manual are not read. Claims needing the manual or customer history may fail
because that information is outside the original lab's prompt.

In [ ]:
def load_data(name):
    return json.loads((ROOT / "data" / name).read_text(encoding="utf-8"))


labels = {
    row["claim_id"]: row for row in load_data("evals/decisions.json") if row["tiny"]
}
claims = [row for row in load_data("claims.json") if row["claim_id"] in labels]
policies = {row["policy_id"]: row for row in load_data("policies.json")}
policy_wording = (ROOT / "data/policy_wording.md").read_text(encoding="utf-8")
assert len(claims) == len(labels) == 5

INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    return f"# Claim\n\n{json.dumps(claim, indent=2)}\n"


def is_right(answer, label):
    return (
        answer["decision"] == label["decision"]
        and abs(answer["payout"] - label["payout"]) <= 1
    )


experiment_inputs = {
    "version": EXPERIMENT_VERSION,
    "models": MODELS,
    "max_model_calls": MAX_MODEL_CALLS,
    "request_interval": REQUEST_INTERVAL,
    "system_prompt": AGENT_SYSTEM_PROMPT,
    "claims": claims,
    "policies": policies,
    "scoring_labels": labels,
}
EXPERIMENT_ID = hashlib.sha256(
    json.dumps(experiment_inputs, sort_keys=True).encode()
).hexdigest()[:16]
EXPECTED_PAIRS = {(model, claim["claim_id"]) for model in MODELS for claim in claims}
print(f"Experiment {EXPERIMENT_ID}: {len(MODELS)} models, {len(claims)} claims")

## The lab's agent

Each claim gets a new agent and a private decision dictionary. Both tools keep
the lab's descriptions and argument order. A five-call limit bounds each run;
an agent that ends without recording this claim is scored as incorrect.
Pacing applies to every internal model request, including calls after tools.

In [ ]:
class RequestPacer:
    def __init__(self, interval=REQUEST_INTERVAL, clock=time.perf_counter, sleep=time.sleep):
        self.interval = interval
        self.clock = clock
        self.sleep = sleep
        self.last_start = None
        self.wait_seconds = 0.0
        self.calls = 0

    def wait(self):
        if self.last_start is not None:
            delay = max(0.0, self.last_start + self.interval - self.clock())
            self.sleep(delay)
            self.wait_seconds += delay
        self.last_start = self.clock()
        self.calls += 1


def build_agent(model, policy_records, system_prompt, pacer):
    recorded = {}

    @tool
    def get_policy(policy_id: str) -> str:
        """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
        policy = policy_records.get(policy_id)
        return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"

    @tool
    def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
        """Record the decision on a claim in the claims system.

        payout is the amount to pay in US dollars, or 0 for a decline or a referral.
        rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
        """
        recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
        return f"Recorded {decision} on {claim_id}"

    @wrap_model_call
    def pace_request(request, handler):
        pacer.wait()
        return handler(request)

    agent = create_agent(
        model=model,
        tools=[get_policy, record_decision],
        system_prompt=system_prompt,
        middleware=[ModelCallLimitMiddleware(run_limit=MAX_MODEL_CALLS), pace_request],
    )
    return agent, recorded


def invoke_claim(model_id, claim, pacer, model=None):
    if model is None:
        model = ChatOpenAI(
            model=model_id,
            base_url=OPENROUTER_BASE_URL,
            api_key=os.environ["OPENROUTER_API_KEY"],
            timeout=120,
            max_retries=0,
            extra_body={"provider": {"require_parameters": True}},
        )
    agent, recorded = build_agent(model, policies, AGENT_SYSTEM_PROMPT, pacer)
    wait_before = pacer.wait_seconds
    calls_before = pacer.calls
    start = time.perf_counter()
    state = agent.invoke(
        {"messages": [{"role": "user", "content": make_claim_prompt(claim)}]},
        config={"recursion_limit": 40},
    )
    elapsed = time.perf_counter() - start
    answer = recorded.get(
        claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""}
    )
    response_models = sorted({
        message.response_metadata["model_name"]
        for message in state["messages"]
        if isinstance(message, AIMessage) and message.response_metadata.get("model_name")
    })
    return {
        "model": model_id,
        "claim_id": claim["claim_id"],
        **answer,
        "seconds": round(elapsed, 3),
        "pacing_seconds": round(pacer.wait_seconds - wait_before, 3),
        "model_calls": pacer.calls - calls_before,
        "response_models": ";".join(response_models),
    }

## Checkpointing and the free allowance

Before each claim, check the remaining free allowance and reserve five requests.
Requests start at least 4.1 seconds apart (under 15 per minute), with SDK retries
disabled. A normal claim takes about three requests, so 15 claims need roughly
45 requests. Extra turns or provider failures may require another day.

A provider or network failure pauses the comparison and leaves that pair pending;
it is not a wrong claims decision. Completed rows are saved atomically after each
claim. Resuming skips those rows. A fingerprint prevents mixing different inputs
or settings; increment `EXPERIMENT_VERSION` if you change the tools or runner.

Model IDs and zero pricing are rechecked against the live catalog. If a chosen
model is unavailable, the run stops rather than substituting another model.
See [OpenRouter's limits](https://openrouter.ai/docs/api_reference/limits).

In [ ]:
RESULT_COLUMNS = [
    "experiment_id", "model", "claim_id", "decision", "payout", "rationale",
    "difficulty", "right", "seconds", "pacing_seconds", "model_calls",
    "response_models", "finished_at",
]


def read_results(path=RESULTS_PATH):
    if not path.exists():
        return []
    with path.open(newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        if reader.fieldnames != RESULT_COLUMNS:
            raise ValueError("Unexpected results columns; use a fresh results.csv.")
        rows = list(reader)
    pairs = set()
    for row in rows:
        pair = row["model"], row["claim_id"]
        if row["experiment_id"] != EXPERIMENT_ID:
            raise ValueError("Saved results belong to different inputs; archive them before rerunning.")
        if pair not in EXPECTED_PAIRS or pair in pairs:
            raise ValueError("Saved results contain an unexpected or duplicate model/claim pair.")
        pairs.add(pair)
        for key in ("payout", "seconds", "pacing_seconds"):
            row[key] = float(row[key])
        row["model_calls"] = int(row["model_calls"])
        row["right"] = is_right(row, labels[row["claim_id"]])
    return rows


def save_results(rows, path=RESULTS_PATH):
    temporary = path.with_suffix(".tmp")
    with temporary.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=RESULT_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


class ExperimentPaused(Exception):
    pass


def validate_catalog(catalog):
    for model_id in MODELS:
        model = catalog.get(model_id)
        if model is None or "tools" not in model.get("supported_parameters", []):
            raise ExperimentPaused(f"Model unavailable or missing tool support: {model_id}")
        pricing = model["pricing"]
        if not model_id.endswith(":free") or any(
            float(pricing.get(key, "0")) != 0 for key in ("prompt", "completion", "request")
        ):
            raise ExperimentPaused(f"Model is not a zero-priced free variant: {model_id}")


def validate_models():
    response = httpx.get(f"{OPENROUTER_BASE_URL}/models", timeout=30)
    response.raise_for_status()
    validate_catalog({row["id"]: row for row in response.json()["data"]})


def remaining_requests():
    response = httpx.get(
        f"{OPENROUTER_BASE_URL}/key",
        headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"},
        timeout=30,
    )
    response.raise_for_status()
    allowance = response.json()["data"].get("free_model_daily_requests")
    if allowance is None:
        raise ExperimentPaused("The API did not report a free request allowance.")
    return allowance["remaining"]


def run_comparison(path=RESULTS_PATH, runner=invoke_claim,
                   allowance=remaining_requests, validate=validate_models):
    rows = read_results(path)
    completed = {(row["model"], row["claim_id"]) for row in rows}
    if completed == EXPECTED_PAIRS:
        return rows
    pacer = RequestPacer()
    try:
        validate()
        for model_id in MODELS:
            for claim in claims:
                pair = model_id, claim["claim_id"]
                if pair in completed:
                    continue
                remaining = allowance()
                if remaining < MAX_MODEL_CALLS:
                    raise ExperimentPaused(f"Only {remaining} free requests remain; need five to start a claim.")
                answer = runner(model_id, claim, pacer)
                label = labels[claim["claim_id"]]
                rows.append({
                    "experiment_id": EXPERIMENT_ID,
                    **answer,
                    "difficulty": label["difficulty"],
                    "right": is_right(answer, label),
                    "finished_at": datetime.now(timezone.utc).isoformat(),
                })
                save_results(rows, path)
                completed.add(pair)
                print(f"{len(completed)}/15: {model_id}, {claim['claim_id']}, right={rows[-1]['right']}")
    except ExperimentPaused as error:
        print(f"Paused: {error}")
    except (APIError, httpx.HTTPError) as error:
        status = getattr(error, "status_code", None)
        if isinstance(error, httpx.HTTPStatusError):
            status = error.response.status_code
        print(f"Paused: {type(error).__name__}, HTTP {status or 'unavailable'}. The current pair remains pending.")
    print(f"Saved {len(rows)}/15 completed pairs. Rerun to resume pending pairs.")
    return rows

## Offline checks

These checks use scripted tool calls and temporary CSVs. They make no network
requests and never add synthetic rows to the published measurements.

In [ ]:
class StubModel(FakeMessagesListChatModel):
    seen: list = Field(default_factory=list)

    def bind_tools(self, tools, **kwargs):
        return self

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        self.seen.extend(messages)
        return super()._generate(messages, stop=stop, run_manager=run_manager, **kwargs)


def check_offline():
    expected = {"decision": "approve", "payout": 100.0}
    assert is_right({"decision": "approve", "payout": 101.0}, expected)
    assert not is_right({"decision": "approve", "payout": 101.01}, expected)
    assert not is_right({"decision": "decline", "payout": 100.0}, expected)

    now = [0.0]
    def sleep(seconds):
        now[0] += seconds

    pacer = RequestPacer(clock=lambda: now[0], sleep=sleep)
    claim = {"claim_id": "CLM-STUB", "policy_id": "POL-STUB"}
    scripted = StubModel(responses=[
        AIMessage(content="", tool_calls=[{
            "name": "get_policy", "args": {"policy_id": "POL-STUB"}, "id": "policy",
        }]),
        AIMessage(content="", tool_calls=[{
            "name": "record_decision", "id": "record", "args": {
                "claim_id": "CLM-STUB", "decision": "approve", "payout": 100.0,
                "rationale": "Synthetic test decision.",
            },
        }]),
        AIMessage(content="Done."),
    ])
    answer = invoke_claim(MODELS[0], claim, pacer, model=scripted)
    assert answer["model"] == MODELS[0] and answer["payout"] == 100.0
    assert answer["model_calls"] == 3 and pacer.wait_seconds == 2 * REQUEST_INTERVAL
    scorer_only = {**expected, "reason": "SCORER_ONLY_TEST_MARKER"}
    assert is_right(answer, scorer_only)
    assert all(scorer_only["reason"] not in str(message.content) for message in scripted.seen)
    missing = invoke_claim(MODELS[1], claim, pacer, model=StubModel(responses=[AIMessage(content="Done.")]))
    assert missing["decision"] == "nothing recorded" and missing["model_calls"] == 1
    looping = StubModel(responses=[AIMessage(content="", tool_calls=[{
        "name": "get_policy", "args": {"policy_id": "POL-STUB"}, "id": "repeat",
    }])])
    capped = invoke_claim(MODELS[2], claim, pacer, model=looping)
    assert capped["model_calls"] == MAX_MODEL_CALLS and capped["decision"] == "nothing recorded"

    calls = []
    def runner(model_id, sample, unused_pacer):
        calls.append((model_id, sample["claim_id"]))
        return {
            **answer, "model": model_id, "claim_id": sample["claim_id"],
            "seconds": 1.0, "pacing_seconds": 0.0,
        }

    with TemporaryDirectory() as temporary:
        path = Path(temporary) / "results.csv"
        assert run_comparison(path, runner, lambda: 2, lambda: None) == []
        assert calls == [] and not path.exists()
        quotas = iter([50, 2])
        partial = run_comparison(path, runner, lambda: next(quotas), lambda: None)
        assert len(partial) == len(read_results(path)) == len(calls) == 1
        def unavailable(model_id, sample, unused_pacer):
            raise httpx.ConnectError("Synthetic offline provider failure")

        interrupted = run_comparison(path, unavailable, lambda: 50, lambda: None)
        assert len(interrupted) == len(read_results(path)) == 1
        resumed = run_comparison(path, runner, lambda: 50, lambda: None)
        assert len(resumed) == len(calls) == len(set(calls)) == 15
        assert run_comparison(path, runner, lambda: 0, lambda: None) == read_results(path)
        assert len(calls) == 15
        resumed[0]["experiment_id"] = "different-inputs"
        save_results(resumed, path)
        try:
            read_results(path)
        except ValueError:
            pass
        else:
            raise AssertionError("Changed inputs must not reuse cached rows.")

    valid_catalog = {model_id: {
        "supported_parameters": ["tools"], "pricing": {"prompt": "0", "completion": "0"},
    } for model_id in MODELS}
    validate_catalog(valid_catalog)
    valid_catalog[MODELS[0]]["pricing"]["completion"] = "0.01"
    try:
        validate_catalog(valid_catalog)
    except ExperimentPaused:
        pass
    else:
        raise AssertionError("A paid variant must not run.")
    print("Offline checks passed: scoring, label isolation, tools, pacing, state isolation, call cap, quota, provider pause, resume, fingerprint and pricing.")


with redirect_stdout(StringIO()) as check_output:
    check_offline()
print(check_output.getvalue().splitlines()[-1])

## Collect or load measurements

The default reads saved measurements without an API key or network access.
For a live run, set `RUN_LIVE = True` in the configuration cell. Run all cells
again after the daily allowance refreshes to continue an incomplete comparison.

In [ ]:
results = run_comparison() if RUN_LIVE else read_results()
print(f"Measured pairs: {len(results)}/15")

## Accuracy and speed

Rank by correct claims only. Equal scores share a rank; speed is reported
separately. Average active seconds exclude deliberate pacing waits but include
the agent's tools and network time. Results are sensitive to free-provider load.
No winner is declared until every model has completed all five claims.

In [ ]:
def summarize(rows):
    summary = []
    for model_id in MODELS:
        measured = [row for row in rows if row["model"] == model_id]
        summary.append({
            "model": model_id,
            "completed": len(measured),
            "correct": sum(row["right"] for row in measured) if measured else None,
            "avg_seconds": sum(row["seconds"] for row in measured) / len(measured) if measured else None,
            "avg_active_seconds": sum(max(0.0, row["seconds"] - row["pacing_seconds"]) for row in measured) / len(measured) if measured else None,
            "model_calls": sum(row["model_calls"] for row in measured),
        })
    table = pd.DataFrame(summary)
    complete = {(row["model"], row["claim_id"]) for row in rows} == EXPECTED_PAIRS
    if complete:
        table["rank"] = table["correct"].rank(method="min", ascending=False).astype(int)
        table = table.sort_values(["rank", "model"])
        winners = table.loc[table["rank"] == 1, "model"].tolist()
        print("Provisional best on these five claims: " + ", ".join(winners))
    else:
        print(f"Comparison pending: {len(rows)}/15 pairs measured. No winner yet.")
    return table.round(3)


display(summarize(results))
misses = [{
    "model": row["model"], "claim_id": row["claim_id"], "difficulty": row["difficulty"],
    "agent_decision": row["decision"], "agent_payout": row["payout"],
    "handler_decision": labels[row["claim_id"]]["decision"],
    "handler_payout": labels[row["claim_id"]]["payout"],
    "agent_rationale": row["rationale"], "handler_reason": labels[row["claim_id"]]["reason"],
} for row in results if not row["right"]]
if misses:
    display(pd.DataFrame(misses))

## Takeaways

Interpret the measured table above, not model size or popularity, to answer the
assignment. Five claims in one run give a provisional comparison, not evidence
of consistent performance across the full queue. The same unavailable manual
or history information affects every candidate. Missing tool decisions are
model failures; provider outages are pending measurements.

To share this work, keep measured rows in `results.csv`, summarize the observed
findings in `README.md`, and clear all notebook outputs and execution counts.
Offline checks use synthetic data only and are never saved to that CSV.